<a href="https://colab.research.google.com/github/Skquark/AEI-Colab-Notebooks/blob/main/MiniMax-H3_ComfyUI_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# 🎬 MiniMax-H3 (ComfyUI runtime) — Video + Audio Generation (33B, INT8 quantized)

A Colab port of [MiniMax-H3](https://huggingface.co/MiniMaxAI/MiniMax-H3) — a 33B parameter
video generation model that produces **video with synchronized audio** (ambience, foley,
speech). Supports text-to-video and image-to-video (first frame).

## How it works

MiniMax-H3 is run as a **headless ComfyUI subprocess** rather than via the stock diffusers
`ModularPipeline` integration:

1. **ComfyUI v0.30.1+ subprocess** on `127.0.0.1:8188`, launched with
   `--disable-pinned-memory --fp16-intermediates`. The pinned-memory flag is the difference
   between an OOM-kill on 32-53 GB host RAM and a 15-second video completing in ~25 min.
   See [`tonyd2wild/minimax-h3-local`](https://github.com/tonyd2wild/minimax-h3-local) for
   the verified recipe.

2. **`Comfy-Org/MiniMax-H3` weights** (`minimax_h3_fl2va_pruned_int8_convrot` diffusion
   transformer + `qwen3vl_32b_minimax_h3_nvfp4_awq` NVFP4 text encoder + `minimax_h3_video_vae_fp16`
   + `minimax_h3_audio_vae_fp32`).

3. **A pure-Python driver** that builds an API-format workflow JSON, POSTs `/prompt`,
   polls `/history/{prompt_id}`, downloads outputs via `/view`. The Gradio UI in STEP 4
   wraps the same driver for interactive use.

```
prompt + first-frame image → ComfyUI subprocess → video.mp4 + audio.wav
                                  (no GPU residency stacking — components swap on each step)
```

## ⚠️ License — Territory Restriction + MAU Cap

MiniMax H3 Community License:
- **Excludes:** EU, UK, South Korea, **and USA**
- **>1M MAU** requires separate commercial license
- Continuing past the header cell is your acceptance of the license

## Quick start

1. **Runtime → Change runtime type → GPU** (L4, A100, A100 80GB, or RTX 3090/4090)
2. Run **STEP 1** — git clones ComfyUI to Drive + installs torch cu130 + requirements
3. Run **STEP 2** — downloads Comfy-Org weights (~40.8 GB total) to Drive cache. First run: 30-60 min.
4. Run **STEP 3** — launches the ComfyUI subprocess and waits for `/system_stats` ready
5. Run **STEP 4** — opens the Gradio UI for t2va / fl2va
6. **STEP 5** keep-alive, **STEP 6** quick test, **STEP 7** batch

## Memory

| GPU | VRAM | Outcome |
|-----|------|---------|
| **A100 80GB** | 80 GB | int8_convrot + NVFP4 + `--disable-pinned-memory` |
| **A100 40GB** | 40 GB | int8_convrot + NVFP4 + `--disable-pinned-memory` |
| **L4 22GB** | 22 GB | int8_convrot + NVFP4 + `--disable-pinned-memory` |
| **RTX 3090/4090** | 24 GB | int8_convrot + NVFP4 + `--disable-pinned-memory` |

Total weight footprint on disk: ~40.8 GB. Fits in 53 GB Colab Pro+ host RAM.

## Outputs

```
<output_dir>/<prefix>_<timestamp>.mp4    # Video + synchronized stereo audio (32 kHz)
```

## Technical notes

- **ComfyUI v0.30.1+** for native H3 node support (`MiniMaxH3ImageToVideo`).
- **`--disable-pinned-memory`** is the single launch flag that prevents the host-RAM OOM-kill
  on 32-53 GB systems.
- **`--fp16-intermediates`** halves inter-node tensor sizes and is cheap insurance.
- **Drive cache**: weights live at `/content/drive/MyDrive/AEI_3D_Cache/H3_ComfyUI/weights/`
  and are symlinked into ComfyUI's `models/` subdirs. Subsequent runs skip the download.
- **`PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`** reduces memory fragmentation on
  ComfyUI's intermediate buffers.

## Companion notebooks

- **`MiniMax-H3_Colab.ipynb`** — the diffusers attempt; left in the repo as documentation
  of the failure modes this notebook's `--disable-pinned-memory` flag sidesteps.
- **Wan2.2_Colab** — text/image-to-video (no audio)
- **Wan2.2_Animate_Colab** — character animation
- **GaussianGPT_Colab** — autoregressive 3D scene generation
- **InfiniSplat_Colab** — single-image 3DGS reconstruction


In [ ]:
#@title STEP 1 — Install ComfyUI v0.30.1+ and dependencies (Drive-persistent)

"""
• Mounts Google Drive for the weights cache + ComfyUI installation
• Clones ComfyUI v0.30.1+ to /content/drive/MyDrive/ComfyUI_H3/ on first run
• Installs torch 2.11.0+cu130 (L4 / T4 / A100 compatible)
• Installs ComfyUI's requirements.txt (transformers, tokenizers, safetensors, av, ...)
• Sets PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to reduce fragmentation
"""

import os, sys, subprocess, time, pathlib
from pathlib import Path

print('='*72)
print('MiniMax-H3 / ComfyUI — Install + Setup')
print('='*72)
try:
    import torch
    print(f'  Python : {sys.version.split()[0]}')
    print(f'  torch  : {torch.__version__}  CUDA: {torch.version.cuda}')
    if torch.cuda.is_available():
        p = torch.cuda.get_device_properties(0)
        print(f'  GPU    : {p.name}  ({p.total_memory / (1024**3):.1f} GB)')
    else:
        print('  WARNING: no GPU detected')
except ImportError:
    print('  torch not yet installed')
print()

CONNECT_GOOGLE_DRIVE = True  #@param {type:'boolean'}
if CONNECT_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive', force_remount=False)
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path('/content/_h3_cache')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

COMFY_DIR = DRIVE_ROOT / 'ComfyUI_H3'
HF_CACHE  = DRIVE_ROOT / 'AEI_3D_Cache' / 'H3_ComfyUI'
OUT_DIR   = DRIVE_ROOT / 'AEI_3D_Out' / 'MiniMax-H3'
COMFY_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)

os.environ['HF_HOME']               = str(HF_CACHE)
os.environ['HUGGINGFACE_HUB_CACHE']  = str(HF_CACHE)
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.8')
os.environ.setdefault('TOKENIZERS_PARALLELISM', 'false')

print(f'  Drive cache  : {HF_CACHE}')
print(f'  ComfyUI dir  : {COMFY_DIR}')
print(f'  Output dir   : {OUT_DIR}')

if not COMFY_DIR.joinpath('main.py').exists():
    print(f'  Cloning ComfyUI to {COMFY_DIR} ...')
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/comfyanonymous/ComfyUI.git', str(COMFY_DIR)], check=True)
else:
    print(f'  Reusing existing {COMFY_DIR}')

# Optimization mode. Standard = the Comfy-Org/MiniMax-H3 int8_convrot base
# at 20 sampling steps (the current behavior). turbo_lora adds the drbaph
# 4-step LoRA + SigmaShift(12, 6) and drops to 8-12 steps (~5x faster).
# turbo_lora_sage layers the kjnodes MiniMaxH3MemoryEfficientSageAttentionPatch
# on top for an additional ~2x. fp8_dense swaps the int8_convrot diffusion
# model for the fp8_scaled one (same 19.5 GB on disk, no INT8 dequant at
# sample time) — faster sampling, better quality at low step counts.
# turbo_lora + fp8_dense compose: prefix mode with `fp8+` to opt in.
# Mode is consumed by STEPs 2/3/4/6/7.
MODE = 'standard'  #@param ['standard', 'turbo_lora', 'turbo_lora_sage', 'fp8_dense', 'fp8_dense+turbo_lora', 'fp8_dense+turbo_lora_sage'] {"allow-input": true}

print('  Installing pytorch cu130 ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                'torch', 'torchvision', 'torchaudio',
                '--index-url', 'https://download.pytorch.org/whl/cu130'], check=False)

print('  Installing ComfyUI requirements ...')
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet',
                '-r', str(COMFY_DIR / 'requirements.txt')], check=False)

# The requirements pull in comfy-kitchen + comfy-aimdo which are no-op stubs for the
# MiniMax-H3 path. We don't need torchao for INT8 (ComfyUI handles it natively).

import torch as _torch_check
print(f'  torch        : {_torch_check.__version__}  (CUDA {_torch_check.version.cuda})')
if _torch_check.cuda.is_available():
    p = _torch_check.cuda.get_device_properties(0)
    print(f'  GPU          : {p.name}  ({p.total_memory / 1024**3:.1f} GB)')
    print(f'  Compute      : {p.major}.{p.minor}')
else:
    raise SystemExit('No GPU detected - ComfyUI needs CUDA.')

# Persist MODE for the rest of the cells (STEPs 2/3/4/6/7 all consume it).
# We use a global dict so that downstream cells can read MODE without
# having to import it (the previous H3_* globals pattern).
_MODE_GLOBAL = globals().setdefault('_H3_MODE', MODE)
import builtins as _builtins
_builtins.H3_MODE = MODE
_MODE_DEFAULT_STEPS = {
    'standard': 20,
    'turbo_lora': 10,
    'turbo_lora_sage': 8,
    'fp8_dense': 16,                 # fp8 has cleaner numerics -> can drop a few steps
    'fp8_dense+turbo_lora': 10,      # same as turbo_lora
    'fp8_dense+turbo_lora_sage': 8,  # same as turbo_lora_sage
}[MODE]
print(f'  Mode         : {MODE}  (default steps: {_MODE_DEFAULT_STEPS})')


In [ ]:
#@title STEP 2 - Download Comfy-Org/MiniMax-H3 weights to Drive cache

import os, time, shutil
from pathlib import Path
from huggingface_hub import snapshot_download, HfApi

# Comfy-Org/MiniMax-H3 layout (4 weights, ~40.8 GB on disk).
#   diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors  (19.5 GB)
#   text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors          (15.7 GB)
#   vae/minimax_h3_video_vae_fp16.safetensors                            ( 5.0 GB)
#   vae/minimax_h3_audio_vae_fp32.safetensors                            ( 0.6 GB)
# Total: ~40.8 GB on disk. The Drive cache keeps them across Colab sessions.
#
# Drive FUSE rejects symlinks and stalls on hardlinks of multi-GB files, so we
# full-copy into ComfyUI's models/ subdirs. We skip the copy when the dst already
# has the exact size from the Comfy-Org manifest (no spurious work on re-runs),
# and we verify sizes after the copy so a corrupted partial file gets caught
# here instead of as a `RuntimeError: shape '[...]' is invalid` deep in
# `comfy/utils.py load_safetensors` at sample time.

WEIGHTS_DIR = Path('/content/drive/MyDrive/AEI_3D_Cache/H3_ComfyUI/weights')
COMFY_DIR = Path('/content/drive/MyDrive/ComfyUI_H3')

# Pull MODE from STEP1's stored builtin (set in STEP1 as H3_MODE).
_MODE = getattr(__import__('builtins'), 'H3_MODE', 'standard')

PATTERNS = [
    'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors',
    'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
    'vae/minimax_h3_video_vae_fp16.safetensors',
    'vae/minimax_h3_audio_vae_fp32.safetensors',
]
# fp8_dense mode swaps the int8_convrot diffusion for the fp8_scaled one
# (added to Comfy-Org/MiniMax-H3 on 2025-08-04). Same 19.5 GB on disk, but
# the kernel runs fp8 matmuls natively so each sampler step is faster and
# the numerics at low step counts are noticeably cleaner. The on-disk
# byte count is identical to int8_convrot (19.5 GB), so Drive quota usage
# is unchanged. Compose with turbo_lora / turbo_lora_sage by prefixing
# the mode with `fp8_dense+` (e.g. `fp8_dense+turbo_lora`).
DIFF_PATTERN = ('diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors'
                if _MODE.startswith('fp8_') else
                'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors')
# Replace the int8 pattern with the fp8 one (they're mutually exclusive
# in PATTERNS — we only ever pull one diffusion file).
PATTERNS = [DIFF_PATTERN if p == 'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors' else p
            for p in PATTERNS]
# Turbo LoRA — drbaph's 4-step pruned checkpoint (~620 MB on disk, Apache 2.0).
# Combined with the Comfy-Org int8 base, this lets the sampler drop to
# 8-12 steps (~5x faster) at the cost of some sampling stability.
# The LoRA is loaded by LoraLoaderModelOnly at strength 1.0.
TURBO_LORA = 'minimax_h3_turbo_4step_pruned_comfyui.safetensors'
TURBO_LORA_PATH = 'loras/' + TURBO_LORA
# The turbo half of the mode (after any fp8_dense+ prefix).
_TURBO_MODE = _MODE.split('+', 1)[1] if '+' in _MODE else _MODE
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage'):
    PATTERNS.append(TURBO_LORA_PATH)

# Resolve expected sizes from the Hub manifest (one HTTP call, ~1s).
print('  Resolving expected sizes from Comfy-Org/MiniMax-H3 ...')
info = HfApi().repo_info('Comfy-Org/MiniMax-H3', files_metadata=True)
EXPECTED_BYTES = {sib.rfilename: sib.size for sib in info.siblings if sib.size is not None}

# When turbo modes are selected we also need the drbaph LoRA from a different
# repo. Merge its expected size so the verify step can validate the bytes,
# and we'll snapshot it separately below.
if _MODE in ('turbo_lora', 'turbo_lora_sage'):
    print('  Resolving Turbo LoRA size from drbaph/MiniMax-H3-Turbo-Lora-ComfyUI ...')
    _tb_info = HfApi().repo_info('drbaph/MiniMax-H3-Turbo-Lora-ComfyUI', files_metadata=True)
    for sib in _tb_info.siblings:
        if sib.rfilename == TURBO_LORA and sib.size is not None:
            EXPECTED_BYTES[TURBO_LORA_PATH] = sib.size

t0 = time.time()
weights = snapshot_download(
    repo_id='Comfy-Org/MiniMax-H3',
    local_dir=str(WEIGHTS_DIR),
    allow_patterns=PATTERNS,
)
print(f'  Weights cached at {weights} in {time.time() - t0:.0f}s')

# Pull the Turbo LoRA from its own repo if it isn't already in WEIGHTS_DIR
# (snapshot_download above only matched Comfy-Org patterns).
if _MODE in ('turbo_lora', 'turbo_lora_sage'):
    _lora_dst = WEIGHTS_DIR / TURBO_LORA_PATH
    if not _lora_dst.exists() or _lora_dst.stat().st_size != EXPECTED_BYTES.get(TURBO_LORA_PATH, -1):
        print(f'  Pulling Turbo LoRA {TURBO_LORA} from drbaph/MiniMax-H3-Turbo-Lora-ComfyUI ...')
        snapshot_download(
            repo_id='drbaph/MiniMax-H3-Turbo-Lora-ComfyUI',
            local_dir=str(WEIGHTS_DIR),
            allow_patterns=[TURBO_LORA_PATH],
        )

DIFF = COMFY_DIR / 'models' / 'diffusion_models'
TXT  = COMFY_DIR / 'models' / 'text_encoders'
VAE  = COMFY_DIR / 'models' / 'vae'
for d in (DIFF, TXT, VAE):
    d.mkdir(parents=True, exist_ok=True)

def _expected_bytes(path_rel: str) -> int:
    return EXPECTED_BYTES.get(path_rel, -1)


def _wire(src: Path, dst: Path, expected_size: int):
    """Copy `src` to `dst` unless `dst` already matches `expected_size`.

    Skips the copy when the existing dst size matches the manifest exactly
    (cheap ~ms check, saves 8-25 min of Drive FUSE copy on re-runs).
    Re-runs that have a partial dst fall through to the full copy.
    Drives a 5-second heartbeat so the user sees progress.
    """
    expected_str = f'{expected_size/1024**3:.2f} GB'
    if dst.exists():
        existing = dst.stat().st_size
        if existing == expected_size:
            print(f'    OK  {dst.name}  ({expected_str} verified)', flush=True)
            return
        # Wrong size: usually a partial write from an interrupted earlier run.
        print(f'    -> {dst.name}: dst is {existing/1024**3:.2f} GB but expected {expected_str}; replacing.', flush=True)
        dst.unlink()
    elif expected_size <= 0:
        print(f'    ?? {dst.name}: no manifest size available; copying without verification.', flush=True)

    src_size_gb = src.stat().st_size / 1024**3
    print(f'    -> {dst.name}  ({src_size_gb:.1f} GB) ...', flush=True)
    t0 = time.time()
    copied = 0
    last_beat = time.time()
    with open(src, 'rb') as fsrc, open(dst, 'wb') as fdst:
        while True:
            chunk = fsrc.read(64 * 1024 * 1024)  # 64 MB chunks
            if not chunk:
                break
            fdst.write(chunk)
            copied += len(chunk)
            if time.time() - last_beat > 5:
                pct = 100 * copied / (src_size_gb * 1024**3)
                rate = copied / (time.time() - t0) / 1024**2
                print(f'       {pct:5.1f}%  {rate:.0f} MB/s', flush=True)
                last_beat = time.time()
    print(f'    -> done in {time.time() - t0:.0f}s', flush=True)


def _verify():
    """Final sanity check: every dst matches the manifest size exactly.
    If anything is off, raise with a delete-and-retry message."""
    bad = []
    for pat in PATTERNS:
        dst = COMFY_DIR / 'models' / pat
        exp = _expected_bytes(pat)
        if not dst.exists():
            bad.append((pat, 0, exp))
            continue
        got = dst.stat().st_size
        if got != exp:
            bad.append((pat, got, exp))
    if bad:
        msg = 'Some weights do not match the Comfy-Org manifest.\n'
        for pat, got, exp in bad:
            msg += f'  {pat}: got {got/1024**3:.2f} GB, expected {exp/1024**3:.2f} GB\n'
        msg += ('Delete the bad files (e.g. '
                '`shutil.rmtree(\'/content/drive/MyDrive/ComfyUI_H3/models\')`) and re-run STEP 2.')
        raise SystemExit(msg)


_wire(WEIGHTS_DIR / DIFF_PATTERN, DIFF / Path(DIFF_PATTERN).name,
         _expected_bytes(DIFF_PATTERN))
_wire(WEIGHTS_DIR / 'text_encoders' / 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
         TXT / 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
         _expected_bytes('text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'))
_wire(WEIGHTS_DIR / 'vae' / 'minimax_h3_video_vae_fp16.safetensors',
         VAE / 'minimax_h3_video_vae_fp16.safetensors',
         _expected_bytes('vae/minimax_h3_video_vae_fp16.safetensors'))
_wire(WEIGHTS_DIR / 'vae' / 'minimax_h3_audio_vae_fp32.safetensors',
         VAE / 'minimax_h3_audio_vae_fp32.safetensors',
         _expected_bytes('vae/minimax_h3_audio_vae_fp32.safetensors'))
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage'):
    LORA_DIR = COMFY_DIR / 'models' / 'loras'
    LORA_DIR.mkdir(parents=True, exist_ok=True)
    _wire(WEIGHTS_DIR / TURBO_LORA_PATH, LORA_DIR / TURBO_LORA,
         _expected_bytes(TURBO_LORA_PATH))

_verify()

print('  Weights wired into ComfyUI/models/{diffusion_models,text_encoders,vae}/ (sizes verified against Comfy-Org/MiniMax-H3 manifest).')
if _MODE.startswith('fp8_'):
    print(f'  fp8_dense: diffusion model is the fp8_scaled checkpoint (no INT8 dequant at sample time).')
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage'):
    print(f'  Turbo LoRA wired into ComfyUI/models/loras/ — strength 1.0 at denoise time.')
total = 0
for d in (DIFF, TXT, VAE):
    for p in sorted(d.iterdir()):
        if p.suffix == '.safetensors':
            sz = p.stat().st_size / 1024**3
            total += sz
            print(f'    {sz:5.2f} GB  {d.name}/{p.name}')
if _TURBO_MODE in ('turbo_lora', 'turbo_lora_sage'):
    for p in sorted(LORA_DIR.iterdir()):
        if p.suffix == '.safetensors':
            sz = p.stat().st_size / 1024**3
            total += sz
            print(f'    {sz:5.2f} GB  loras/{p.name}')
print(f'  Total on ComfyUI side: {total:.2f} GB')


In [ ]:
#@title STEP 3 - Launch ComfyUI subprocess (--disable-pinned-memory)

import os, sys, time, signal, subprocess, urllib.request, urllib.error, json
from pathlib import Path

COMFY_DIR = Path('/content/drive/MyDrive/ComfyUI_H3')
COMFY_HOST = '127.0.0.1'
COMFY_PORT = 8188
COMFY_URL  = f'http://{COMFY_HOST}:{COMFY_PORT}'

# Pinned memory is the entire reason this notebook exists: the default ComfyUI policy pins
# ~90% of host RAM. With 53 GB of RAM the pinned ceiling is ~47 GB, and the diffusers runtime
# OOM-kills long before the model fits. --disable-pinned-memory is the single flag that makes
# this recipe work on 24-32 GB RAM (see tonyd2wild/minimax-h3-local).
# --fp16-intermediates halves inter-node tensors and is cheap insurance.
LAUNCH_CMD = [
    sys.executable, 'main.py',
    '--listen', COMFY_HOST,
    '--port', str(COMFY_PORT),
    '--disable-pinned-memory',
    '--fp16-intermediates',
    '--disable-api-nodes',
    '--output-directory', str(COMFY_DIR / 'output'),
    '--input-directory', str(COMFY_DIR / 'input'),
]
env = os.environ.copy()

subprocess.run(['pkill', '-9', '-f', 'ComfyUI_H3.*main.py'], check=False)
time.sleep(2)

(COMFY_DIR / 'output').mkdir(parents=True, exist_ok=True)
(COMFY_DIR / 'input').mkdir(parents=True, exist_ok=True)

# Optional: install kjnodes custom node if MODE = turbo_lora_sage. kjnodes
# provides the MiniMaxH3MemoryEfficientSageAttentionPatch node that drops
# attention memory ~2x. We skip the install for other modes since the
# 620 MB of upstream deps would be wasted.
_MODE3_pre = getattr(__import__('builtins'), 'H3_MODE', 'standard')
_TURBO_MODE3_pre = _MODE3_pre.split('+', 1)[1] if '+' in _MODE3_pre else _MODE3_pre
if _TURBO_MODE3_pre == 'turbo_lora_sage':
    _CUSTOM = COMFY_DIR / 'custom_nodes' / 'ComfyUI-KJNodes'
    if not _CUSTOM.exists():
        print('  Cloning ComfyUI-KJNodes (for SageAttention) ...')
        subprocess.run(['git', 'clone', '--depth=1',
                        'https://github.com/kijai/ComfyUI-KJNodes.git', str(_CUSTOM)], check=True)
        req = _CUSTOM / 'requirements.txt'
        if req.exists():
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', '-r', str(req)], check=False)
    else:
        print(f'  Reusing existing {_CUSTOM}')

print(f'  Launching: {" ".join(LAUNCH_CMD)}')
log_path = COMFY_DIR / 'comfyui.log'
log_f = open(log_path, 'wb')
proc = subprocess.Popen(
    LAUNCH_CMD,
    cwd=str(COMFY_DIR),
    stdout=log_f,
    stderr=subprocess.STDOUT,
    env=env,
    preexec_fn=os.setsid,
)
print(f'  PID: {proc.pid}, log: {log_path}')

ready = False
deadline = time.time() + 600
while time.time() < deadline:
    try:
        with urllib.request.urlopen(COMFY_URL + '/system_stats', timeout=2) as r:
            r.read()
        ready = True
        break
    except (urllib.error.URLError, ConnectionResetError, OSError):
        if proc.poll() is not None:
            print(f'  ComfyUI exited early with code {proc.returncode}. Last 40 log lines:')
            with open(log_path) as f:
                tail = f.read().splitlines()[-40:]
                for ln in tail:
                    print('   ', ln)
            raise SystemExit('ComfyUI failed to start.')
    time.sleep(2)

if not ready:
    proc.terminate()
    raise SystemExit(f'ComfyUI did not respond within 10 minutes. Tail of log:\n'
                     + '\n'.join(open(log_path).read().splitlines()[-20:]))

# Sanity check the model files BEFORE we hand the workflow off. ComfyUI's
# load_torch_file reads each tensor with `view(info["shape"])`, so a partial
# file whose metadata header says 33.5M elements but whose bytes are only
# 11.9M will throw `RuntimeError: shape '[...]' is invalid` deep inside the
# load. Catching it here instead of at denoise time is the difference
# between "fast feedback" and "45 min of waiting for the sampler to fail".
from huggingface_hub import HfApi as _HfApi
_HF_API = _HfApi()
_INFO = _HF_API.repo_info('Comfy-Org/MiniMax-H3', files_metadata=True)
_EXPECTED = {sib.rfilename: sib.size for sib in _INFO.siblings if sib.size is not None}
_MODE3 = getattr(__import__('builtins'), 'H3_MODE', 'standard')
_TURBO_MODE3 = _MODE3.split('+', 1)[1] if '+' in _MODE3 else _MODE3
if _TURBO_MODE3 in ('turbo_lora', 'turbo_lora_sage'):
    # Pull the LoRA's expected size from its own repo (drbaph/MiniMax-H3-Turbo-Lora-ComfyUI).
    # This repo is much smaller than Comfy-Org/MiniMax-H3 so the API call is cheap.
    _TURBO_INFO = _HF_API.repo_info('drbaph/MiniMax-H3-Turbo-Lora-ComfyUI', files_metadata=True)
    for sib in _TURBO_INFO.siblings:
        if sib.rfilename == TURBO_LORA and sib.size is not None:
            _EXPECTED['loras/' + TURBO_LORA] = sib.size
# Pick the diffusion file based on MODE: fp8_scaled for fp8_* modes,
# int8_convrot for the rest. The text encoder + VAE patterns are unchanged.
_DIFF_PAT3 = ('diffusion_models/minimax_h3_fl2va_pruned_fp8_scaled.safetensors'
              if _MODE3.startswith('fp8_') else
              'diffusion_models/minimax_h3_fl2va_pruned_int8_convrot.safetensors')
_PATTERNS = [
    _DIFF_PAT3,
    'text_encoders/qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors',
    'vae/minimax_h3_video_vae_fp16.safetensors',
    'vae/minimax_h3_audio_vae_fp32.safetensors',
]
if _TURBO_MODE3 in ('turbo_lora', 'turbo_lora_sage'):
    _PATTERNS.append('loras/' + TURBO_LORA)
print('  Verifying ComfyUI/models file sizes against Comfy-Org/MiniMax-H3 manifest ...')
_size_bad = []
for pat in _PATTERNS:
    dst = COMFY_DIR / 'models' / pat
    exp = _EXPECTED.get(pat, -1)
    if not dst.exists():
        _size_bad.append((pat, 0, exp))
        print(f'    MISSING  {dst.name}  (expected {exp/1024**3:.2f} GB)')
        continue
    got = dst.stat().st_size
    if got != exp:
        _size_bad.append((pat, got, exp))
        print(f'    WRONG    {dst.name}  (got {got/1024**3:.2f} GB, expected {exp/1024**3:.2f} GB)')
    else:
        print(f'    OK       {dst.name}  ({exp/1024**3:.2f} GB)')
if _size_bad:
    proc.terminate()
    msg = ('ComfyUI-side weights do not match the Comfy-Org manifest. '
           'This usually means a Drive FUSE copy was interrupted before completion.\n\n')
    for pat, got, exp in _size_bad:
        msg += f'  {pat}: got {got/1024**3:.2f} GB, expected {exp/1024**3:.2f} GB\n'
    msg += ('\nFix: delete the bad models and re-run STEP 2:\n'
            '  !rm -rf /content/drive/MyDrive/ComfyUI_H3/models\n'
            '  # then re-run STEP 2 (it skips files that match the manifest size).\n')
    raise SystemExit(msg)

with urllib.request.urlopen(COMFY_URL + '/system_stats') as r:
    sysinfo = json.loads(r.read())
print(f'  ComfyUI ready: {sysinfo.get("system", {}).get("comfyui_version", "?")} on '
      f'{sysinfo.get("devices", [{}])[0].get("name", "?")}')

import builtins
builtins.H3_COMFY_PROC = proc
builtins.H3_COMFY_LOG = log_path
builtins.H3_COMFY_URL = COMFY_URL
builtins.H3_COMFY_DIR = COMFY_DIR

import re as _re
def _slug(s, maxlen=40):
    """Compress a prompt to a filesystem-safe slug, capped at maxlen chars.

    Strips non-alphanumerics, collapses runs of separators, lowercases. The
    result is short enough to fit a ComfyUI SaveVideo prefix and stays
    human-readable in the output filename.
    """
    s = _re.sub(r'[^a-zA-Z0-9_\-]+', '-', s).strip('-')
    s = _re.sub(r'-+', '-', s)
    return s[:maxlen].strip('-') or 'untitled'

builtins.H3_SLUG = _slug
print('  Stored H3_COMFY_PROC / H3_COMFY_URL in builtins for STEP 4 access.')


In [ ]:
#@title STEP 4 - Gradio UI: t2v / fl2v with audio, runs through ComfyUI /prompt + /history

import os, json, time, uuid, shutil, urllib.request, urllib.parse, builtins
from pathlib import Path

COMFY_URL  = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')
COMFY_DIR  = getattr(builtins, 'H3_COMFY_DIR', Path('/content/drive/MyDrive/ComfyUI_H3'))
PROC       = getattr(builtins, 'H3_COMFY_PROC', None)

import requests

def _poll_until_ready(url, timeout=10):
    deadline = time.time() + timeout
    while time.time() < deadline:
        try:
            with urllib.request.urlopen(url, timeout=2) as r:
                r.read()
            return True
        except Exception:
            time.sleep(1)
    return False

if not _poll_until_ready(COMFY_URL + '/system_stats'):
    raise SystemExit('ComfyUI is not running. Re-run STEP 3.')

def _build_workflow(mode, prompt, width, height, length, steps, seed,
                    unet_name, clip_name, video_vae, audio_vae, first_frame=None,
                    last_frame=None, filename_prefix=None, turbo=False, sage=False):
    p = {}
    p["6"]  = {"class_type": "UNETLoader",   "inputs": {"unet_name": unet_name, "weight_dtype": "default"}}
    p["13"] = {"class_type": "CLIPLoader",   "inputs": {"clip_name": clip_name, "type": "minimax", "device": "default"}}
    p["11"] = {"class_type": "VAELoader",    "inputs": {"vae_name": video_vae}}
    p["24"] = {"class_type": "VAELoader",    "inputs": {"vae_name": audio_vae}}
    if mode == 'fl2v' and first_frame is not None:
        p["30"] = {"class_type": "LoadImage", "inputs": {"image": first_frame}}
    if mode == 'fl2v' and last_frame is not None:
        p["31"] = {"class_type": "LoadImage", "inputs": {"image": last_frame}}
    inputs = {"clip": ["13", 0], "vae": ["11", 0],
              "width": width, "height": height, "length": length, "prompt": prompt}
    if mode == 'fl2v' and first_frame is not None:
        inputs["first_frame"] = ["30", 0]
    if mode == 'fl2v' and last_frame is not None:
        inputs["last_frame"] = ["31", 0]
    p["104"] = {"class_type": "MiniMaxH3ImageToVideo", "inputs": inputs}

    # MODEL chain: UNETLoader -> LoraLoader (if turbo) -> SigmaShift (if turbo) -> SageAttention (if turbo_sage) -> consumers
    _model_out = ["6", 0]
    if turbo:
        p["40"] = {"class_type": "LoraLoaderModelOnly",
                   "inputs": {"model": _model_out, "lora_name": TURBO_LORA, "strength": 1.0}}
        _model_out = ["40", 0]
        # MiniMaxH3SigmaShift: video_shift=12, audio_shift=6 are the upstream-recommended values
        # for the drbaph turbo_lora (per the example workflow we vendored from).
        p["41"] = {"class_type": "MiniMaxH3SigmaShift",
                   "inputs": {"model": _model_out, "shift_video": 12.0, "shift_audio": 6.0}}
        _model_out = ["41", 0]
        if sage:
            # KJNodes' SageAttention patch. Running after SigmaShift keeps the
            # LoRA-modified model's sigmas intact while still patching the
            # attention ops. sage_attention=2 selects the kjnodes default
            # which is auto-detected at install time (sparse if available).
            p["42"] = {"class_type": "MiniMaxH3MemoryEfficientSageAttentionPatch",
                       "inputs": {"model": _model_out, "sage_attention": 2}}
            _model_out = ["42", 0]

    p["16"] = {"class_type": "BasicGuider",   "inputs": {"model": _model_out, "conditioning": ["104", 0]}}
    p["17"] = {"class_type": "KSamplerSelect","inputs": {"sampler_name": "res_multistep"}}
    p["9"]  = {"class_type": "BasicScheduler","inputs": {"model": _model_out, "scheduler": "simple",
                                                          "steps": steps, "denoise": 1}}
    p["15"] = {"class_type": "RandomNoise",   "inputs": {"noise_seed": seed}}
    p["14"] = {"class_type": "SamplerCustomAdvanced",
               "inputs": {"noise": ["15", 0], "guider": ["16", 0], "sampler": ["17", 0],
                          "sigmas": ["9", 0], "latent_image": ["104", 1]}}
    p["10"] = {"class_type": "VAEDecode",      "inputs": {"samples": ["14", 0], "vae": ["11", 0]}}
    p["23"] = {"class_type": "VAEDecodeAudio", "inputs": {"samples": ["14", 0], "vae": ["24", 0]}}
    p["91"] = {"class_type": "CreateVideo",    "inputs": {"images": ["10", 0], "audio": ["23", 0], "fps": 24}}
    p["92"] = {"class_type": "SaveVideo",      "inputs": {"video": ["91", 0],
                                                            "filename_prefix": filename_prefix or f"video/H3_{mode}",
                                                           "format": "auto", "codec": "auto"}}
    return {"prompt": p}


def _upload_image(local_path):
    with open(local_path, 'rb') as f:
        files = {'image': (Path(local_path).name, f, 'image/png')}
        data  = {'type': 'input', 'overwrite': 'true'}
        r = requests.post(COMFY_URL + '/upload/image', files=files, data=data, timeout=120)
    r.raise_for_status()
    return r.json()['name']


def _queue_workflow(wf, client_id):
    r = requests.post(COMFY_URL + '/prompt', json={"prompt": wf["prompt"], "client_id": client_id}, timeout=60)
    if r.status_code != 200:
        raise RuntimeError(f'ComfyUI rejected workflow: {r.status_code} {r.text[:500]}')
    return r.json()['prompt_id']


def _wait_for_history(prompt_id, timeout=3600, poll=3):
    deadline = time.time() + timeout
    while time.time() < deadline:
        r = requests.get(f'{COMFY_URL}/history/{prompt_id}', timeout=10)
        if r.status_code == 200 and prompt_id in r.json():
            return r.json()[prompt_id]
        time.sleep(poll)
    raise TimeoutError(f'Workflow {prompt_id} did not finish within {timeout}s')


def _download_outputs(history_entry, out_dir):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)
    paths = []
    for node_id, node_out in history_entry.get('outputs', {}).items():
        for kind in ('videos', 'images', 'audio'):
            for entry in node_out.get(kind, []):
                fn = entry.get('filename')
                if not fn:
                    continue
                subdir = entry.get('type', 'output')
                # ComfyUI writes outputs to --output-directory synchronously
                # and reports the same filename in /history. The output file
                # is usually already on disk by the time the workflow is
                # reported as `completed` — read it directly from out_dir
                # instead of going through /view (Drive FUSE adds latency
                # to /view lookups and 404s while the FUSE cache catches up).
                #
                # ComfyUI's SaveVideo with a `filename_prefix` containing a
                # `/` separator creates the output under a subdirectory, but
                # the /history `filename` strips the directory prefix. So
                # both `out_dir/<fn>` and `out_dir/<dir>/<fn>` may exist.
                # Probe the most-likely locations in order.
                candidates = [out_dir / fn]
                if '/' not in fn:
                    # ComfyUI stripped the directory prefix from the
                    # filename; the file may be under any subdirectory
                    # that was part of our workflow's filename_prefix.
                    # Probe the common case (`video/`) plus a recursive
                    # glob for safety.
                    candidates.append(out_dir / 'video' / fn)
                # Wait up to 12 s for the FUSE write-flush to make any
                # candidate visible. Check every 2 s.
                local = None
                for _wait in range(7):
                    for c in candidates:
                        if c.exists():
                            local = c
                            break
                    if local is not None:
                        break
                    # Fallback: scan one level down for the filename.
                    if local is None and _wait == 3:
                        for hit in out_dir.glob(f'*/{fn}'):
                            local = hit
                            break
                    time.sleep(2)
                if local is None:
                    print(f'    Local file not at {[str(c) for c in candidates]} after 14 s, falling back to /view ...', flush=True)
                    src_url = f'{COMFY_URL}/view?filename={urllib.parse.quote(fn)}&type={subdir}'
                    local = candidates[0]
                    for _attempt in range(3):
                        try:
                            with urllib.request.urlopen(src_url, timeout=120) as r, open(local, 'wb') as f:
                                shutil.copyfileobj(r, f)
                            break
                        except urllib.error.HTTPError as e:
                            if e.code == 404 and _attempt < 2:
                                time.sleep(3)
                                continue
                            raise
                paths.append(str(local))
    return paths


import gradio as gr

OUTPUT_DIR = COMFY_DIR / 'output'

def _fmt(seconds):
    """Format a duration in seconds as `H:MM:SS` (or `MM:SS` under an hour).

    Used by the Gradio progress bar and the STEP 6 polling loop so the user can
    read elapsed time at a glance when a 124-frame H3 generation takes ~30-50 min.
    """
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'

def run_minimax_h3(mode, prompt, first_frame, last_frame,
                    canvas_label, length, steps, seed,
                    unet_name, clip_name, video_vae, audio_vae,
                    progress=gr.Progress(track_tqdm=False)):
    if not prompt or not prompt.strip():
        raise gr.Error('MiniMax-H3 needs a non-empty prompt.')
    # Seed 0 (or negative) = random. Matches the convention in STEP 6/7.
    if seed is None or int(seed) <= 0:
        seed = random.randint(1, 2**31 - 1)
        progress(0.0, desc=f'Random seed: {seed}')
    seed = int(seed)
    # Resolve canvas dims from the dropdown label; fall back to the first
    # known entry if the user typed a stale label.
    if canvas_label not in CANVASES:
        canvas_label = next(iter(CANVASES))
    width, height = CANVASES[canvas_label]
    while length % 17 != 5:
        length += 1

    client_id = str(uuid.uuid4())
    first_frame_name = None
    last_frame_name = None
    if mode == 'fl2v' and first_frame is not None:
        first_frame_name = _upload_image(first_frame)
    if mode == 'fl2v' and last_frame is not None:
        last_frame_name = _upload_image(last_frame)

    # Build a self-describing filename: video/H3_<mode>_<slug>_<WxH>_f<length>_s<steps>_seed<seed>_<ts>
    # ComfyUI appends `_` + epoch-ms + counter, so the full filename ends up
    # around 90-120 chars — well within filesystem limits.
    slug = builtins.H3_SLUG(prompt, maxlen=40)
    ts = int(time.time())
    prefix = (f'video/H3_{mode}_{slug}_{int(width)}x{int(height)}_f{int(length)}'
              f'_s{int(steps)}_seed{int(seed)}_{ts}')

    # Composed mode (e.g. `fp8_dense+turbo_lora`) is split: the fp8 part
    # already happened at UNET-loader time, the turbo/sage part happens here.
    _turbo_side = _MODE_FOR_STEPS.split('+', 1)[1] if '+' in _MODE_FOR_STEPS else _MODE_FOR_STEPS
    wf = _build_workflow(mode, prompt, int(width), int(height), int(length), int(steps), int(seed),
                          unet_name, clip_name, video_vae, audio_vae,
                          first_frame=first_frame_name,
                          last_frame=last_frame_name,
                          filename_prefix=prefix,
                          turbo=_turbo_side != 'standard',
                          sage=_turbo_side == 'turbo_lora_sage')
    progress(0.05, desc=f'Queueing {mode} workflow ({width}x{height}, {length} frames, {steps} steps)...')
    prompt_id = _queue_workflow(wf, client_id)
    progress(0.1, desc=f'Queued as {prompt_id[:8]}. Polling ComfyUI queue...')

    started = time.time()
    while True:
        h = _wait_for_history(prompt_id, timeout=120)
        if h.get('status', {}).get('completed'):
            break
        if h.get('status', {}).get('error'):
            raise gr.Error('ComfyUI failed: ' + json.dumps(h['status'].get('messages', []), indent=2)[:2000])
        elapsed = time.time() - started
        progress(min(0.1 + 0.85 * (elapsed / 600), 0.95),
                 desc=f'Denoising in progress... {_fmt(elapsed)} elapsed ({(elapsed / max(int(steps),1)):.1f}s/step est.)')

    paths = _download_outputs(h, OUTPUT_DIR)
    progress(1.0, desc=f'Done in {_fmt(time.time() - started)}. Outputs: {[Path(p).name for p in paths]}')
    if not paths:
        raise gr.Error('Workflow finished but no outputs were produced.')
    video_path = next((p for p in paths if p.endswith(('.mp4', '.webm', '.mov'))), paths[0])
    report = (f'`{width}x{height}`, {length} frames ({length/24:.3f}s), {steps} steps - '
              f'time {time.time() - started:.0f}s - seed {seed} - mode `{mode}`')
    return video_path, report


CANVASES = [
    # 16:9 landscape — the trained default
    ('1152 x 640 · 16:9 wide',           1152, 640),
    ('960 x 544 · 16:9 standard',         960, 544),
    ('832 x 480 · 16:9 fast (default)',   832, 480),
    ('704 x 400 · 16:9 cheaper',          704, 400),
    ('576 x 320 · 16:9 minimal',          576, 320),
    # 9:16 portrait — TikTok / Shorts / Reels
    ('640 x 1152 · 9:16 wide',            640, 1152),
    ('544 x 960 · 9:16 standard',         544, 960),
    ('480 x 832 · 9:16 fast',             480, 832),
    ('400 x 704 · 9:16 cheaper',          400, 704),
    ('320 x 576 · 9:16 minimal',          320, 576),
    # 1:1 square
    ('768 x 768 · 1:1 full',              768, 768),
    ('576 x 576 · 1:1 fast',              576, 576),
    # 4:3 / 3:4
    ('768 x 576 · 4:3',                   768, 576),
    ('576 x 768 · 3:4',                   576, 768),
    # Cinematic wide
    ('1344 x 544 · ~21:9',                1344, 544),
    ('1024 x 416 · ~21:9 cheaper',        1024, 416),
]
DEFAULT_CANVAS = '832 x 480 · 16:9 fast (default)'
DEFAULT_LENGTH = 124  # 5 s at 24 fps

UNET = ('minimax_h3_fl2va_pruned_fp8_scaled.safetensors'
        if getattr(__import__('builtins'), 'H3_MODE', 'standard').startswith('fp8_') else
        'minimax_h3_fl2va_pruned_int8_convrot.safetensors')
CLIP = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
VVAE = 'minimax_h3_video_vae_fp16.safetensors'
AVAE = 'minimax_h3_audio_vae_fp32.safetensors'

_MODE4_BADGE = getattr(__import__('builtins'), 'H3_MODE', 'standard')
_MODE4_NOTE = {
    'standard': '20 steps, int8_convrot diffusion, baseline.',
    'turbo_lora': '10 steps, int8 + drbaph 4-step LoRA + SigmaShift(12, 6).',
    'turbo_lora_sage': '8 steps, int8 + LoRA + SigmaShift + KJNodes SageAttention.',
    'fp8_dense': '16 steps, fp8_scaled diffusion (cleaner numerics).',
    'fp8_dense+turbo_lora': '10 steps, fp8 + LoRA + SigmaShift.',
    'fp8_dense+turbo_lora_sage': '8 steps, fp8 + LoRA + SigmaShift + SageAttention.',
}.get(_MODE4_BADGE, 'mode: ' + _MODE4_BADGE)
with gr.Blocks(title='MiniMax-H3 (ComfyUI)') as demo:
    gr.Markdown('# MiniMax-H3 - joint video + audio\n\nBackend: ComfyUI subprocess (--disable-pinned-memory). Weights: Comfy-Org/MiniMax-H3 (' + UNET.replace('minimax_h3_fl2va_pruned_', '').replace('.safetensors', '') + ' diffusion + NVFP4 text encoder).')
    gr.Markdown(f'**Active mode:** `{_MODE4_BADGE}` — {_MODE4_NOTE}')
    gr.Markdown('### Welcome\n\nMiniMax-H3 generates 5-15 s of synchronized video + stereo audio from a prompt or first frame. The model is hosted by a ComfyUI subprocess running on 127.0.0.1:8188 (started in STEP 3).\n\n**First run:** keep the defaults (832 x 480, 124 frames = 5 s, 20 steps).')
    with gr.Row():
        with gr.Column():
            mode = gr.Radio(choices=[('Text-to-video (t2va)', 't2va'),
                                      ('Image-to-video (fl2va)', 'fl2va')],
                            value='t2va', label='Mode',
                            info='t2va is the most reliable; fl2va needs an explicit first frame.')
            prompt = gr.Textbox(lines=6,
                                value=('Cinematic medium shot of a red fox trotting through a snowy pine forest at dawn, '
                                       'snow crunching underfoot, natural handheld micro-movement, shallow depth of field, '
                                       '35mm film grain.\n\noverall_soundscape: forest ambience, distant birdsong, '
                                       'footfalls on frozen snow.\nnon_diegetic_music: none.'),
                                label='Prompt',
                                info='Supports [Shot N] scene direction and Dialogue: lines. Inline dialogue is spoken by the audio decoder.')
            with gr.Row():
                first_frame = gr.Image(label='First frame (fl2va)', type='filepath', sources=['upload'])
                last_frame  = gr.Image(label='Last frame (fl2va)',  type='filepath', sources=['upload'])
            canvas_dd = gr.Dropdown(choices=[c[0] for c in CANVASES], value=DEFAULT_CANVAS, label='Canvas',
                                    info='Smaller canvas + shorter length = dramatically faster.')
            length = gr.Slider(56, 362, value=DEFAULT_LENGTH, step=1, label='Length (frames at 24 fps)',
                               info='Snapped to the 17n+5 grid. 124 ~ 5 s, 362 ~ 15 s.')
            with gr.Row():
                # Steps slider range depends on MODE: 10-40 for standard,
                # 4-16 for turbo (since 8-12 is the sweet spot for the LoRA).
                _gr_mode = getattr(__import__('builtins'), 'H3_MODE', 'standard')
                _gr_min = 10 if _gr_mode == 'standard' else 4
                _gr_max = 40 if _gr_mode == 'standard' else 16
                _gr_def = {
                    'standard': 20, 'turbo_lora': 10, 'turbo_lora_sage': 8,
                    'fp8_dense': 16, 'fp8_dense+turbo_lora': 10, 'fp8_dense+turbo_lora_sage': 8,
                }[_gr_mode]
                steps = gr.Slider(_gr_min, _gr_max, value=_gr_def, step=1, label='Steps',
                                  info='20 is the upstream default; 24-28 marginally improves quality. Turbo LoRAs need 8-12.')
                seed  = gr.Number(value=42, precision=0, label='Seed',
                                  info='0 = random.')
            run = gr.Button('Generate', variant='primary')
        with gr.Column():
            video_out = gr.Video(label='Video + soundtrack')
            report    = gr.Markdown()

    run.click(
        run_minimax_h3,
        [mode, prompt, first_frame, last_frame, canvas_dd, length, steps, seed,
         gr.State(UNET), gr.State(CLIP), gr.State(VVAE), gr.State(AVAE)],
        [video_out, report],
    )

    def _welcome():
        return (
            'ComfyUI is running. Submit a workflow via the Gradio UI on '
            'http://127.0.0.1:7860 or POST to /prompt with an API-format workflow JSON.',
        )
    demo.load(_welcome, None, [report])

demo.queue(default_concurrency_limit=1).launch(share=False, inline=False, prevent_thread_lock=True, server_port=7860)
import builtins
builtins.H3_DEMO = demo
from IPython.display import clear_output as _clear
_clear()
print('\nGradio UI: http://127.0.0.1:7860 (open the Colab proxy URL above)')


In [ ]:
#@title STEP 5 - Keep-alive + session summary (ComfyUI status)

import time, json, urllib.request, builtins

URL = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')

import IPython.display
display(IPython.display.Javascript("""
function KeepAlive() { console.log('Colab session kept alive at ' + new Date().toISOString()); }
setInterval(KeepAlive, 60000);
"""))

print('=' * 72)
print('MiniMax-H3 / ComfyUI session summary')
print('=' * 72)

try:
    with urllib.request.urlopen(URL + '/system_stats', timeout=5) as r:
        stats = json.loads(r.read())
    devs = stats.get('devices', [])
    for d in devs:
        free  = d.get('vram_free', 0) / 1024**3
        total = d.get('vram_total', 0) / 1024**3
        print(f'  GPU {d.get("name"):<24} {free:5.1f} GB free / {total:5.1f} GB total')
    print(f'  ComfyUI version  : {stats.get("system", {}).get("comfyui_version", "?")}')
    print(f'  Python version   : {stats.get("system", {}).get("python_version", "?")}')
    print(f'  Embedded at      : {URL}')
    print(f'  Log file         : {getattr(builtins, "H3_COMFY_LOG", "?")}')
    print(f'  Output dir       : {getattr(builtins, "H3_COMFY_DIR", "?")}/output')
    print()
    print('  ComfyUI subprocess is running. Three ways to use it:')
    print('    - STEP 4 (Gradio UI): interactive t2va / fl2va at http://127.0.0.1:7860')
    print('    - STEP 6 (quick test): single video via form params, polls /history')
    print('    - STEP 7 (batch): JSON scene list at /content/drive/MyDrive/AEI_3D_Cache/H3_ComfyUI/batch_scenes.json')
    print('  STEP 8 tails the ComfyUI log if a run stalls.')
except Exception as e:
    print(f'  [WARN] ComfyUI not reachable: {e}')


In [ ]:
#@title STEP 6 — Quick test (single video generation)

"""Stand-alone test. Generates one video with the parameters below."""
import os, sys, time, json, pathlib, random, uuid, shutil, urllib.request, urllib.parse, urllib.error, requests
from pathlib import Path
import builtins
from IPython.display import display, FileLink

URL = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')
OUT_DIR = Path(getattr(builtins, 'H3_COMFY_DIR', Path('/content/drive/MyDrive/ComfyUI_H3'))) / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('='*72)
print('MiniMax-H3 / ComfyUI — single-video quick test')
print('='*72)

PROMPT = 'Cinematic medium shot of a red fox trotting through a snowy pine forest at dawn, snow crunching underfoot, natural handheld micro-movement, shallow depth of field, 35mm film grain.\n\noverall_soundscape: forest ambience, distant birdsong, footfalls on frozen snow.\nnon_diegetic_music: none.'  #@param {type:"string"}
IMAGE_PATH = ''  #@param {type:"string"}
LAST_IMAGE_PATH = ''  #@param {type:"string"}
# Canvas label must match a key in CANVASES below. See STEP 4 (Gradio UI)
# for the full dropdown; this is a free-text field so it stays in sync.
CANVAS = "832 x 480 · 16:9 fast (default)"  #@param {type:"string"}
DURATION = 5  #@param {type:"slider", min:2, max:14, step:1}
# Default steps depend on MODE: 20 for standard, 10 for turbo_lora,
# 8 for turbo_lora_sage. The slider max is 20 for standard and 16 for
# turbo modes since you're rarely above 12 steps with the LoRA.
_MODE_FOR_STEPS = getattr(__import__('builtins'), 'H3_MODE', 'standard')
# Split composed mode into fp8 prefix + turbo suffix for the steps map.
_TURBO_FOR_STEPS = _MODE_FOR_STEPS.split('+', 1)[1] if '+' in _MODE_FOR_STEPS else _MODE_FOR_STEPS
_DEFAULT_STEPS = {
    'standard': 20, 'turbo_lora': 10, 'turbo_lora_sage': 8,
    'fp8_dense': 16, 'fp8_dense+turbo_lora': 10, 'fp8_dense+turbo_lora_sage': 8,
}[_MODE_FOR_STEPS]
_STEPS_MAX = 40 if _MODE_FOR_STEPS == 'standard' else 16
STEPS = _DEFAULT_STEPS  #@param {type:"slider", min:4, max:40, step:1}
SEED = 42  #@param {type:"integer"}

# Canvas presets (must match the list in STEP 4 / Gradio dropdown).
CANVASES = {
    # 16:9 landscape
    '1152 x 640 · 16:9 wide':           (1152, 640),
    '960 x 544 · 16:9 standard':        (960, 544),
    '832 x 480 · 16:9 fast (default)':  (832, 480),
    '704 x 400 · 16:9 cheaper':         (704, 400),
    '576 x 320 · 16:9 minimal':         (576, 320),
    # 9:16 portrait
    '640 x 1152 · 9:16 wide':           (640, 1152),
    '544 x 960 · 9:16 standard':        (544, 960),
    '480 x 832 · 9:16 fast':            (480, 832),
    '400 x 704 · 9:16 cheaper':         (400, 704),
    '320 x 576 · 9:16 minimal':         (320, 576),
    # 1:1 square
    '768 x 768 · 1:1 full':             (768, 768),
    '576 x 576 · 1:1 fast':             (576, 576),
    # 4:3 / 3:4
    '768 x 576 · 4:3':                  (768, 576),
    '576 x 768 · 3:4':                  (576, 768),
    # Cinematic wide
    '1344 x 544 · ~21:9':               (1344, 544),
    '1024 x 416 · ~21:9 cheaper':       (1024, 416),
}
# Validate canvas label — fall back to the first known entry if the user
# typed something that doesn't exist (CANVASES is the source of truth).
if CANVAS not in CANVASES:
    fallback = next(iter(CANVASES))
    print(f'  Canvas label {CANVAS!r} not in CANVASES; falling back to {fallback!r}')
    CANVAS = fallback
WIDTH, HEIGHT = CANVASES[CANVAS]
LENGTH = DURATION * 24
while LENGTH % 17 != 5:
    LENGTH += 1
if SEED <= 0:
    SEED = random.randint(1, 2**31 - 1)
    print(f'  Random seed: {SEED}')

img_path = IMAGE_PATH.strip() or None
last_img_path = LAST_IMAGE_PATH.strip() or None

print(f'  Prompt      : {PROMPT[:60]}...')
print(f'  Canvas      : {WIDTH} x {HEIGHT}  ({LENGTH} frames = {LENGTH / 24:.3f}s at 24 fps)')
print(f'  Steps       : {STEPS}')
print(f'  Seed        : {SEED}')
print(f'  First frame : {img_path or "(none)"}')
print(f'  Last frame  : {last_img_path or "(none)"}')
print()

UNET = ('minimax_h3_fl2va_pruned_fp8_scaled.safetensors'
        if getattr(__import__('builtins'), 'H3_MODE', 'standard').startswith('fp8_') else
        'minimax_h3_fl2va_pruned_int8_convrot.safetensors')
CLIP = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
VVAE = 'minimax_h3_video_vae_fp16.safetensors'
AVAE = 'minimax_h3_audio_vae_fp32.safetensors'

# Build the workflow. fl2va if a first or last frame is given; else t2va.
inputs = {'clip': ['13', 0], 'vae': ['11', 0], 'width': WIDTH, 'height': HEIGHT,
          'length': LENGTH, 'prompt': PROMPT}
load_image_nodes = {}
if img_path:
    # POST /upload/image to get a server-side filename
    with open(img_path, 'rb') as f:
        files = {'image': (Path(img_path).name, f, 'image/png')}
        data  = {'type': 'input', 'overwrite': 'true'}
        r = requests.post(URL + '/upload/image', files=files, data=data, timeout=120)
    r.raise_for_status()
    server_name = r.json()['name']
    inputs['first_frame'] = ['30', 0]
    load_image_nodes['30'] = {'class_type': 'LoadImage', 'inputs': {'image': server_name}}
if last_img_path:
    # Last-frame input uses node id 31 in upstream MiniMaxH3ImageToVideo
    # workflows. If only last_frame is set (no first_frame), MiniMaxH3ImageToVideo
    # still picks the fl2va branch because last_frame is set.
    with open(last_img_path, 'rb') as f:
        files = {'image': (Path(last_img_path).name, f, 'image/png')}
        data  = {'type': 'input', 'overwrite': 'true'}
        r = requests.post(URL + '/upload/image', files=files, data=data, timeout=120)
    r.raise_for_status()
    server_name = r.json()['name']
    inputs['last_frame'] = ['31', 0]
    load_image_nodes['31'] = {'class_type': 'LoadImage', 'inputs': {'image': server_name}}
load_image_node = load_image_nodes or None

# Self-describing filename: video/H3_quicktest_<slug>_<WxH>_f<length>_s<steps>_seed<seed>_<ts>
# ComfyUI appends `_` + epoch-ms + counter, so the full filename ends up
# around 90-120 chars — well within filesystem limits.
slug = _slug(PROMPT, maxlen=40)
ts = int(time.time())
_filename_prefix = (f'video/H3_quicktest_{slug}_{WIDTH}x{HEIGHT}_f{LENGTH}'
                    f'_s{STEPS}_seed{SEED}_{ts}')

nodes = {}
nodes.update(load_image_node or {})
nodes.update({
    '6':  {'class_type': 'UNETLoader',     'inputs': {'unet_name': UNET, 'weight_dtype': 'default'}},
    '13': {'class_type': 'CLIPLoader',     'inputs': {'clip_name': CLIP, 'type': 'minimax', 'device': 'default'}},
    '11': {'class_type': 'VAELoader',      'inputs': {'vae_name': VVAE}},
    '24': {'class_type': 'VAELoader',      'inputs': {'vae_name': AVAE}},
    '104':{'class_type': 'MiniMaxH3ImageToVideo', 'inputs': inputs},
    '16': {'class_type': 'BasicGuider',    'inputs': {'model': ['6', 0], 'conditioning': ['104', 0]}},
    '17': {'class_type': 'KSamplerSelect', 'inputs': {'sampler_name': 'res_multistep'}},
    '9':  {'class_type': 'BasicScheduler', 'inputs': {'model': ['6', 0], 'scheduler': 'simple', 'steps': STEPS, 'denoise': 1}},
    '15': {'class_type': 'RandomNoise',    'inputs': {'noise_seed': SEED}},
    '14': {'class_type': 'SamplerCustomAdvanced',
           'inputs': {'noise': ['15', 0], 'guider': ['16', 0], 'sampler': ['17', 0],
                      'sigmas': ['9', 0], 'latent_image': ['104', 1]}},
    '10': {'class_type': 'VAEDecode',      'inputs': {'samples': ['14', 0], 'vae': ['11', 0]}},
    '23': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['14', 0], 'vae': ['24', 0]}},
    '91': {'class_type': 'CreateVideo',    'inputs': {'images': ['10', 0], 'audio': ['23', 0], 'fps': 24}},
    '92': {'class_type': 'SaveVideo',      'inputs': {'video': ['91', 0],
                                                        'filename_prefix': _filename_prefix,
                                                        'format': 'auto', 'codec': 'auto'}},
})

t0 = time.time()
r = requests.post(URL + '/prompt', json={'prompt': nodes, 'client_id': str(uuid.uuid4())}, timeout=60)
if r.status_code != 200:
    raise SystemExit(f'ComfyUI rejected the workflow: {r.status_code} {r.text[:1000]}')
prompt_id = r.json()['prompt_id']
print(f'  Queued: {prompt_id[:8]} ... polling /history')

last_report = 0
last_log_size = 0
log_path = Path(getattr(builtins, 'H3_COMFY_LOG', '/content/drive/MyDrive/ComfyUI_H3/comfyui.log'))


def _slug(s, maxlen=40):
    """Compress a prompt to a filesystem-safe slug, capped at maxlen chars.

    Strips non-alphanumerics, collapses runs of separators, lowercases. The
    result is short enough to fit a ComfyUI SaveVideo prefix and stays
    human-readable in the output filename.
    """
    import re as _re
    s = _re.sub(r'[^a-zA-Z0-9_\-]+', '-', s).strip('-')
    s = _re.sub(r'-+', '-', s)
    return s[:maxlen].strip('-') or 'untitled'


def _fmt(seconds):
    """Format a duration as `H:MM:SS` (or `MM:SS` under an hour).

    Duplicated in STEP 4 (Gradio UI) so each cell is self-contained per
    Colab convention — running STEP 6 before STEP 4 should not break it.
    """
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'
while True:
    h = requests.get(f'{URL}/history/{prompt_id}', timeout=10).json()
    if prompt_id in h:
        entry = h[prompt_id]
        if entry.get('status', {}).get('completed'):
            elapsed = time.time() - t0
            print(f'  Done in {_fmt(elapsed)} ({(elapsed / STEPS):.1f}s/step).')
            break
        if entry.get('status', {}).get('error'):
            raise SystemExit('ComfyUI failed: ' + json.dumps(entry['status'].get('messages', []), indent=2)[:2000])
    elapsed = time.time() - t0
    # Pulse print strategy:
    #   - Immediately on any new log line (this is when the user learns
    #     something useful: sampling-step progress, VAE decode messages,
    #     model-load events).
    #   - Every 5 min as a "still alive" indicator so the user knows the
    #     kernel hasn't died if ComfyUI is quiet for a while.
    new_lines = []
    if log_path.exists():
        cur_size = log_path.stat().st_size
        if cur_size > last_log_size:
            with log_path.open('rb') as f:
                f.seek(last_log_size)
                new_lines = f.read().decode('utf-8', errors='replace').splitlines()
            last_log_size = cur_size
    tail = ' | '.join(new_lines[-3:])[:200] if new_lines else ''
    if tail or time.time() - last_report > 300:
        last_report = time.time()
        print(f'    ... {_fmt(elapsed)} elapsed' + (f'  log: {tail}' if tail else ''))
    time.sleep(3)

paths = []
for node_id, node_out in entry.get('outputs', {}).items():
    for kind in ('videos', 'images', 'audio'):
        for out in node_out.get(kind, []):
            fn = out.get('filename')
            if not fn:
                continue
            subdir = out.get('type', 'output')
            fn = out.get('filename')
            if not fn:
                continue
            # ComfyUI's SaveVideo strips the directory prefix from the
            # filename reported in /history. Probe `OUT_DIR/<fn>` and
            # `OUT_DIR/video/<fn>` since our prefix starts with `video/`.
            candidates = [OUT_DIR / fn]
            if '/' not in fn:
                candidates.append(OUT_DIR / 'video' / fn)
            local = None
            for _wait in range(7):
                for c in candidates:
                    if c.exists():
                        local = c
                        break
                if local is not None:
                    break
                if _wait == 3 and local is None:
                    for hit in OUT_DIR.glob(f'*/{fn}'):
                        local = hit
                        break
                time.sleep(2)
            if local is None:
                print(f'    Local file not at {[str(c) for c in candidates]} after 14 s, falling back to /view ...', flush=True)
                # Fall back to /view — works on local fs, may 404 briefly on FUSE.
                local = candidates[0]
                url = f'{URL}/view?filename={urllib.parse.quote(fn)}&type={subdir}'
                for attempt in range(3):
                    try:
                        with urllib.request.urlopen(url, timeout=120) as r, open(local, 'wb') as f:
                            shutil.copyfileobj(r, f)
                        break
                    except urllib.error.HTTPError as e:
                        if e.code == 404 and attempt < 2:
                            time.sleep(3)
                            continue
                        raise
            paths.append(local)

print(f'\n  Outputs ({len(paths)}):')
for p in paths:
    print(f'    {p}')
video_path = next((str(p) for p in paths if str(p).endswith(('.mp4', '.webm', '.mov'))), str(paths[0]))
print(f'\n  Open: {video_path}')
display(FileLink(video_path))


In [ ]:
#@title STEP 7 — Batch generation from a JSON scene list

"""
Advanced batch processor. Reads a JSON file containing a list of scenes, each with its
own prompt, optional keyframe images, canvas, duration, steps, and seed. The ComfyUI
subprocess is shared across scenes; each scene is a separate workflow.

JSON format (a list of objects):
```json
[
  {
    "prompt": "A red fox in a snowy pine forest at dawn",
    "image": "/content/drive/MyDrive/keyframes/fox_start.png",
    "last_image": "/content/drive/MyDrive/keyframes/fox_end.png",
    "canvas": "832 x 480 · 16:9",
    "duration": 5,
    "steps": 20,
    "seed": 42
  }
]
```

Fields (all optional except `prompt`):
  - prompt:    (required) text description of the scene
  - image:     (optional) path to first frame image
  - canvas:    (optional, default DEFAULT_CANVAS) one of the CANVASES keys
  - duration:  (optional, default DEFAULT_DURATION) seconds (2-14)
  - steps:     (optional, default DEFAULT_STEPS) inference steps (10-40)
  - seed:      (optional, default 0 = random) 0 or negative = random

A progress log is written to batch_log.jsonl so you can resume after a disconnect.
If the JSON file does not exist yet, this cell writes a starter template.
"""

import os, sys, time, json, pathlib, random, uuid, urllib.request, urllib.parse, requests, shutil, gc
from pathlib import Path
import builtins

URL = getattr(builtins, 'H3_COMFY_URL', 'http://127.0.0.1:8188')
OUT_DIR = Path(getattr(builtins, 'H3_COMFY_DIR', Path('/content/drive/MyDrive/ComfyUI_H3'))) / 'output'
OUT_DIR.mkdir(parents=True, exist_ok=True)


def _fmt(seconds):
    """Format a duration as `H:MM:SS` (or `MM:SS` under an hour).

    Duplicated in STEP 4 (Gradio UI) and STEP 6 (quick test) so each cell
    is self-contained per Colab convention.
    """
    seconds = int(seconds)
    h, rem = divmod(seconds, 3600)
    m, s = divmod(rem, 60)
    return f'{h}:{m:02d}:{s:02d}' if h else f'{m:02d}:{s:02d}'

print('='*72)
print('MiniMax-H3 / ComfyUI — Batch generation (JSON scene list)')
print('='*72)

BATCH_JSON_PATH = '/content/drive/MyDrive/AEI_3D_Cache/H3_ComfyUI/batch_scenes.json'  #@param {type:"string"}
DEFAULT_DURATION = 5  #@param {type:"slider", min:2, max:14, step:1}
# Default steps scale with MODE (lower for turbo paths). The slider is also
# widened down to 4 to support turbo step counts of 6-12.
_MODE_FOR_STEPS7 = getattr(__import__('builtins'), 'H3_MODE', 'standard')
_DEFAULT_STEPS7 = {
    'standard': 20, 'turbo_lora': 10, 'turbo_lora_sage': 8,
    'fp8_dense': 16, 'fp8_dense+turbo_lora': 10, 'fp8_dense+turbo_lora_sage': 8,
}[_MODE_FOR_STEPS7]
DEFAULT_STEPS = _DEFAULT_STEPS7  #@param {type:"slider", min:4, max:40, step:1}
# Canvas label must match a key in the CANVASES dict below. See STEP 4
# (Gradio UI) for the full dropdown; this is a free-text field so STEP 6/7
# stay in sync with the dict.
DEFAULT_CANVAS = "832 x 480 · 16:9 fast (default)"  #@param {type:"string"}
SKIP_EXISTING = True  #@param {type:"boolean"}

CANVASES = {
    # 16:9 landscape
    '1152 x 640 · 16:9 wide':           (1152, 640),
    '960 x 544 · 16:9 standard':        (960, 544),
    '832 x 480 · 16:9 fast (default)':  (832, 480),
    '704 x 400 · 16:9 cheaper':         (704, 400),
    '576 x 320 · 16:9 minimal':         (576, 320),
    # 9:16 portrait
    '640 x 1152 · 9:16 wide':           (640, 1152),
    '544 x 960 · 9:16 standard':        (544, 960),
    '480 x 832 · 9:16 fast':            (480, 832),
    '400 x 704 · 9:16 cheaper':         (400, 704),
    '320 x 576 · 9:16 minimal':         (320, 576),
    # 1:1 square
    '768 x 768 · 1:1 full':             (768, 768),
    '576 x 576 · 1:1 fast':             (576, 576),
    # 4:3 / 3:4
    '768 x 576 · 4:3':                  (768, 576),
    '576 x 768 · 3:4':                  (576, 768),
    # Cinematic wide
    '1344 x 544 · ~21:9':               (1344, 544),
    '1024 x 416 · ~21:9 cheaper':       (1024, 416),
}

json_path = Path(BATCH_JSON_PATH)
if not json_path.exists():
    json_path.parent.mkdir(parents=True, exist_ok=True)
    template = [
        {'prompt': 'A red fox in a snowy pine forest at dawn, slow dolly push-in, snow crunching underfoot.',
         'canvas': DEFAULT_CANVAS, 'duration': DEFAULT_DURATION, 'steps': DEFAULT_STEPS, 'seed': 42},
        {'prompt': 'A busy night market, neon signs reflecting in puddles, sizzling street food, ambient chatter.',
         'canvas': DEFAULT_CANVAS, 'duration': DEFAULT_DURATION, 'steps': DEFAULT_STEPS, 'seed': 7},
        {'prompt': 'A cellist playing a slow melody in an empty concert hall, warm stage lighting, distant applause at the end.',
         'canvas': DEFAULT_CANVAS, 'duration': DEFAULT_DURATION, 'steps': DEFAULT_STEPS, 'seed': 99},
    ]
    json_path.write_text(json.dumps(template, indent=2))
    print(f'  Created batch {json_path} with {len(template)} starter scenes.')
    print(f'  Edit it, then re-run STEP 7.')

with json_path.open() as f:
    scenes = json.load(f)
if not isinstance(scenes, list):
    raise SystemExit(f'Expected a JSON list, got {type(scenes).__name__}')
print(f'  Loaded {len(scenes)} scene(s) from {json_path}')


def _slug(s, maxlen=40):
    """Compress a prompt to a filesystem-safe slug, capped at maxlen chars.

    Strips non-alphanumerics, collapses runs of separators, lowercases. The
    result is short enough to fit a ComfyUI SaveVideo prefix and stays
    human-readable in the output filename.
    """
    import re as _re
    s = _re.sub(r'[^a-zA-Z0-9_\-]+', '-', s).strip('-')
    s = _re.sub(r'-+', '-', s)
    return s[:maxlen].strip('-') or 'untitled'

UNET = ('minimax_h3_fl2va_pruned_fp8_scaled.safetensors'
        if getattr(__import__('builtins'), 'H3_MODE', 'standard').startswith('fp8_') else
        'minimax_h3_fl2va_pruned_int8_convrot.safetensors')
CLIP = 'qwen3vl_32b_minimax_h3_nvfp4_awq.safetensors'
VVAE = 'minimax_h3_video_vae_fp16.safetensors'
AVAE = 'minimax_h3_audio_vae_fp32.safetensors'

results = []
total_start = time.time()
for i, sc in enumerate(scenes):
    prompt = sc.get('prompt', '').strip()
    if not prompt:
        print(f'  [{i}] SKIP: empty prompt')
        continue
    canvas_label = sc.get('canvas', DEFAULT_CANVAS)
    w, h = CANVASES.get(canvas_label, (832, 480))
    duration = int(sc.get('duration', DEFAULT_DURATION))
    length = duration * 24
    while length % 17 != 5:
        length += 1
    steps = int(sc.get('steps', DEFAULT_STEPS))
    seed = int(sc.get('seed', 0))
    if seed <= 0:
        seed = random.randint(1, 2**31 - 1)
    img_path = sc.get('image', '').strip() or None

    # Self-describing per-scene filename. The scene index keeps the order
    # visible in `ls -t`, the slug makes the filename readable, the
    # canvas/length/steps/seed lets you see at a glance what the run
    # was, and the timestamp suffix prevents collisions across re-runs.
    scene_slug = _slug(prompt, maxlen=30)
    scene_ts = int(time.time())
    scene_prefix = (f'video/H3_batch_{i:03d}_{scene_slug}_{w}x{h}_f{length}'
                    f'_s{steps}_seed{seed}_{scene_ts}')

    # SKIP_EXISTING — check whether a previously-completed run for this scene
    # already wrote the output. We glob on the scene index prefix which
    # doesn't include the timestamp suffix (different runs overwrite by
    # design — the timestamp ensures the new file is unique).
    if SKIP_EXISTING:
        existing = list(OUT_DIR.glob(f'H3_batch_{i:03d}_*'))
        if existing:
            print(f'  [{i+1}/{len(scenes)}] SKIP (already exists: {existing[0].name})')
            results.append(str(existing[0]))
            continue

    print(f'\n  [{i+1}/{len(scenes)}] {w}x{h} {length}f {steps}steps seed={seed}')
    print(f'    {prompt[:80]}')

    inputs = {'clip': ['13', 0], 'vae': ['11', 0], 'width': w, 'height': h,
              'length': length, 'prompt': prompt}
    load_image_node = None
    if img_path:
        with open(img_path, 'rb') as f:
            files = {'image': (Path(img_path).name, f, 'image/png')}
            data  = {'type': 'input', 'overwrite': 'true'}
            r = requests.post(URL + '/upload/image', files=files, data=data, timeout=120)
        r.raise_for_status()
        inputs['first_frame'] = ['30', 0]
        load_image_node = {'30': {'class_type': 'LoadImage', 'inputs': {'image': r.json()['name']}}}

    nodes = {}
    nodes.update(load_image_node or {})
    nodes.update({
        '6':  {'class_type': 'UNETLoader',   'inputs': {'unet_name': UNET, 'weight_dtype': 'default'}},
        '13': {'class_type': 'CLIPLoader',   'inputs': {'clip_name': CLIP, 'type': 'minimax', 'device': 'default'}},
        '11': {'class_type': 'VAELoader',    'inputs': {'vae_name': VVAE}},
        '24': {'class_type': 'VAELoader',    'inputs': {'vae_name': AVAE}},
        '104':{'class_type': 'MiniMaxH3ImageToVideo', 'inputs': inputs},
        '16': {'class_type': 'BasicGuider',   'inputs': {'model': ['6', 0], 'conditioning': ['104', 0]}},
        '17': {'class_type': 'KSamplerSelect','inputs': {'sampler_name': 'res_multistep'}},
        '9':  {'class_type': 'BasicScheduler','inputs': {'model': ['6', 0], 'scheduler': 'simple', 'steps': steps, 'denoise': 1}},
        '15': {'class_type': 'RandomNoise',   'inputs': {'noise_seed': seed}},
        '14': {'class_type': 'SamplerCustomAdvanced',
               'inputs': {'noise': ['15', 0], 'guider': ['16', 0], 'sampler': ['17', 0],
                          'sigmas': ['9', 0], 'latent_image': ['104', 1]}},
        '10': {'class_type': 'VAEDecode',      'inputs': {'samples': ['14', 0], 'vae': ['11', 0]}},
        '23': {'class_type': 'VAEDecodeAudio', 'inputs': {'samples': ['14', 0], 'vae': ['24', 0]}},
        '91': {'class_type': 'CreateVideo',    'inputs': {'images': ['10', 0], 'audio': ['23', 0], 'fps': 24}},
        '92': {'class_type': 'SaveVideo',      'inputs': {'video': ['91', 0],
                                                            'filename_prefix': scene_prefix,
                                                            'format': 'auto', 'codec': 'auto'}},
    })

    t0 = time.time()
    r = requests.post(URL + '/prompt', json={'prompt': nodes, 'client_id': str(uuid.uuid4())}, timeout=60)
    if r.status_code != 200:
        print(f'    FAIL: {r.status_code} {r.text[:300]}')
        continue
    pid = r.json()['prompt_id']
    print(f'    Queued as {pid[:8]} — polling /history for completion ...')
    last_report = 0
    last_log_size = 0
    log_path = Path(getattr(builtins, 'H3_COMFY_LOG', '/content/drive/MyDrive/ComfyUI_H3/comfyui.log'))
    while True:
        h = requests.get(f'{URL}/history/{pid}', timeout=10).json()
        if pid in h:
            entry = h[pid]
            if entry.get('status', {}).get('completed'):
                elapsed = time.time() - t0
                print(f'    Done in {_fmt(elapsed)} ({(elapsed / steps):.1f}s/step).')
                break
            if entry.get('status', {}).get('error'):
                print(f'    FAIL: {json.dumps(entry["status"].get("messages", []))[:300]}')
                break
        elapsed = time.time() - t0
        # Pulse print strategy:
        #   - Immediately on any new log line (sampling-step progress,
        #     VAE decode messages, model-load events).
        #   - Every 5 min as a "still alive" indicator for quiet stretches.
        new_lines = []
        if log_path.exists():
            cur_size = log_path.stat().st_size
            if cur_size > last_log_size:
                with log_path.open('rb') as f:
                    f.seek(last_log_size)
                    new_lines = f.read().decode('utf-8', errors='replace').splitlines()
                last_log_size = cur_size
        tail = ' | '.join(new_lines[-3:])[:200] if new_lines else ''
        if tail or time.time() - last_report > 300:
            last_report = time.time()
            print(f'    ... {_fmt(elapsed)} elapsed' + (f'  log: {tail}' if tail else ''), flush=True)
        time.sleep(5)
    elapsed = time.time() - t0
    for node_id, node_out in entry.get('outputs', {}).items():
        for v in node_out.get('videos', []):
            fn = v['filename']
            local_candidates = [OUT_DIR / fn]
            if '/' not in fn:
                local_candidates.append(OUT_DIR / 'video' / fn)
            # Prefer the local file (already on disk by the time /history
            # reports completed). Fall back to /view only if FUSE has not
            # yet flushed the file metadata; that 404 was the previous run's
            # failure mode.
            local = None
            for _wait in range(7):
                for c in local_candidates:
                    if c.exists():
                        local = c
                        break
                if local is not None:
                    break
                if _wait == 3 and local is None:
                    for hit in OUT_DIR.glob(f'*/{fn}'):
                        local = hit
                        break
                time.sleep(2)
            if local is None:
                local = local_candidates[0]
                url = f'{URL}/view?filename={urllib.parse.quote(fn)}&type=output'
                for _attempt in range(3):
                    try:
                        with urllib.request.urlopen(url, timeout=120) as r2, open(local, 'wb') as f:
                            shutil.copyfileobj(r2, f)
                        break
                    except urllib.error.HTTPError as e:
                        if e.code == 404 and _attempt < 2:
                            time.sleep(3)
                            continue
                        raise
            results.append(str(local))
            print(f'    {local.name}  ({_fmt(elapsed)})')
    gc.collect()

total = time.time() - total_start
print(f'\nBatch complete: {len(results)}/{len(scenes)} clips, total {_fmt(total)} ({total / max(len(results),1):.0f}s/clip)')
for p in results:
    print(f'  {p}')


In [ ]:
#@title STEP 8 — Tail ComfyUI log (debugging aid)

import builtins
from pathlib import Path

LOG = Path(getattr(builtins, 'H3_COMFY_LOG', '/content/drive/MyDrive/ComfyUI_H3/comfyui.log'))
TAIL_LINES = 80  #@param {type:"slider", min:20, max:500, step:20}

if not LOG.exists():
    print(f'  Log not found: {LOG}')
else:
    lines = LOG.read_text(errors='replace').splitlines()
    n = min(TAIL_LINES, len(lines))
    print(f'=== last {n} lines of {LOG} ===')
    for ln in lines[-n:]:
        print(ln)
    print('=== end ===')
